In [7]:
import numpy as np
import pandas as pd


RUHRGEBIET_STAEDTE = [
    "Bochum", "Bottrop", "Dortmund", "Duisburg", "Gelsenkirchen",
    "Hagen", "Hamm", "Herne", "Mülheim an der Ruhr", "Oberhausen", "Essen"]

COL_CITY = "Name"
COL_DENS  = "Bevölkerungsdichte"
COL_SHARE = "Anteil Kinder a.d. Gesamtbev."
COL_POP   = "Bevölkerung 6 bis 20"

# CSV laden + Ruhrgebiet filtern
df = pd.read_csv("../data/processed/master_2024.csv")

d = df.copy()
ruhr = d[d[COL_CITY].isin(RUHRGEBIET_STAEDTE)].copy()
ruhr[COL_SHARE] = ruhr[COL_SHARE] / 100.0


#Essen-Vektor holen + Vergleichsstädte vorbereiten
feat_cols = [COL_DENS, COL_SHARE, COL_POP]
essen_rows = ruhr[ruhr[COL_CITY] == "Essen"].copy()

comp = ruhr[ruhr[COL_CITY] != "Essen"].dropna(subset=feat_cols).copy()
essen_rows["na_cnt"] = essen_rows[feat_cols].isna().sum(axis=1)
essen = essen_rows.sort_values("na_cnt").iloc[0]
essen_vec = essen[feat_cols].astype(float)

# Ähnlichkeit per z-standardisierter euklidischer Distanz
mu = comp[feat_cols].mean()
sd = comp[feat_cols].std(ddof=0).replace(0, np.nan)

comp_z = (comp[feat_cols] - mu) / sd
essen_z = (essen_vec - mu) / sd

comp["dist_z_euclid"] = np.sqrt(((comp_z - essen_z) ** 2).sum(axis=1))

ranking_z = comp[[COL_CITY] + feat_cols + ["dist_z_euclid"]].sort_values("dist_z_euclid")
print(ranking_z.head(3))



# "Sehr ähnlich" filtern (Schwelle Distanz 1)

similar_z = ranking_z[ranking_z["dist_z_euclid"] <= 1.0]
print(similar_z)


        Name  Bevölkerungsdichte  Anteil Kinder a.d. Gesamtbev.  \
34  Dortmund              2149.8                          0.135   
35  Duisburg              2157.1                          0.142   
31    Bochum              2462.4                          0.121   

    Bevölkerung 6 bis 20  dist_z_euclid  
34                 81355       0.897039  
35                 71210       1.458359  
31                 43266       2.261806  
        Name  Bevölkerungsdichte  Anteil Kinder a.d. Gesamtbev.  \
34  Dortmund              2149.8                          0.135   

    Bevölkerung 6 bis 20  dist_z_euclid  
34                 81355       0.897039  
